# ESLint Comment-to-Code Ratio — Raw Output Extraction (JavaScript)

This notebook analyzes **JavaScript repositories** with **ESLint** and derives **Comment-to-Code Ratio** from extracted comment and code line metrics alongside ESLint findings.

**Default benchmark repository:** [expressjs/express](https://github.com/expressjs/express)

> **Note:** Comment-to-Code Ratio is a **Derived** metric. ESLint does not emit it directly; the notebook parses JavaScript sources for comment/code lines and runs ESLint for documentation and maintainability rules.


## Section 1 — Install Dependencies

Install Python packages and verify Node.js / ESLint.


In [ ]:
!pip install -q pandas gitpython jupyter

import shutil, subprocess, sys

for cmd in [['node', '--version'], ['npm', '--version']]:
    subprocess.run(cmd, check=False)

if not shutil.which('eslint'):
    !npm install -g eslint

subprocess.run(['eslint', '--version'], check=False)


## Section 2 — Configuration


In [ ]:
USE_GIT_URL = True

REPO_URL = 'https://github.com/expressjs/express.git'

LOCAL_REPO_PATH = '/content/express'

WORKSPACE_DIR = './workspace'

OUTPUT_DIR = './outputs'

IF_CLONE_EXISTS = 'reuse'

CLONE_DEPTH = 1

RAW_OUTPUT_PREVIEW_LINES = 150

from pathlib import Path

METRIC_ROOT = Path('.').resolve()
PROJECT_ROOT = METRIC_ROOT
for _ in range(8):
    runtimes = PROJECT_ROOT / 'runtimes'
    if runtimes.is_dir() and (runtimes / 'node_modules').is_dir():
        break
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
RUNTIMES_ROOT = PROJECT_ROOT / 'runtimes'

# Fast validation benchmark:
# USE_GIT_URL = False
# LOCAL_REPO_PATH = './workspace/comment_to_code_ratio_benchmark'


## Section 3 — Imports and Utility Functions


In [ ]:
from pathlib import Path
import sys

TOOL_ROOT = Path('tool').resolve()
if str(TOOL_ROOT) not in sys.path:
    sys.path.insert(0, str(TOOL_ROOT))

from __future__ import annotations

import csv
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display
from git import Repo
from git.exc import GitCommandError, InvalidGitRepositoryError

EXCLUDED_DIR_NAMES = {
    ".git", "node_modules", "dist", "build", "coverage", "vendor", "docs", "test", "tests",
}


class NotebookLogger:
    def __init__(self, error_log_path: Path) -> None:
        self.error_log_path = error_log_path
        self.error_log_path.parent.mkdir(parents=True, exist_ok=True)
        self._errors: list[dict[str, str]] = []
        if not self.error_log_path.exists():
            self.write_errors()

    def info(self, message: str) -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        print(f"[{timestamp}] INFO: {message}")

    def error(self, message: str, file: str = "notebook") -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        line = f"[{timestamp}] ERROR: {message}\n"
        print(line, end="")
        self._errors.append({"timestamp": timestamp, "file": file, "error_message": message})
        self.write_errors()

    def write_errors(self) -> None:
        with self.error_log_path.open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=["timestamp", "file", "error_message"])
            writer.writeheader()
            writer.writerows(self._errors)


def ensure_output_dir(output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)


def derive_clone_path(repo_url: str, workspace_dir: Path) -> Path:
    repo_name = repo_url.rstrip("/").removesuffix(".git").split("/")[-1]
    if not repo_name:
        raise ValueError(f"Unable to derive repository name from URL: {repo_url}")
    return workspace_dir / repo_name


def validate_repo_url(repo_url: str) -> None:
    if not repo_url or not isinstance(repo_url, str):
        raise ValueError("REPO_URL must be a non-empty string.")
    if not (repo_url.startswith("https://") or repo_url.startswith("git@") or repo_url.startswith("http://")):
        raise ValueError(f"Invalid repository URL format: {repo_url}")


def clone_or_reuse_repository(
    repo_url: str, workspace_dir: Path, if_clone_exists: str, logger: NotebookLogger, clone_depth: int | None = None,
) -> Path:
    validate_repo_url(repo_url)
    workspace_dir.mkdir(parents=True, exist_ok=True)
    clone_path = derive_clone_path(repo_url, workspace_dir)
    if clone_path.exists():
        if if_clone_exists == "reclone":
            logger.info(f"Removing existing clone at {clone_path}")
            shutil.rmtree(clone_path)
        elif if_clone_exists == "reuse":
            logger.info(f"Reusing existing clone at {clone_path}")
            return clone_path.resolve()
        else:
            raise ValueError("IF_CLONE_EXISTS must be 'reuse' or 'reclone'.")
    logger.info(f"Cloning {repo_url} into {clone_path}")
    clone_kwargs: dict[str, Any] = {"depth": clone_depth} if clone_depth else {}
    try:
        Repo.clone_from(repo_url, clone_path, **clone_kwargs)
    except GitCommandError as exc:
        logger.error(f"Git clone failed: {exc}", file=repo_url)
        raise
    return clone_path.resolve()


def validate_local_repo_path(local_repo_path: Path, logger: NotebookLogger) -> Path:
    if not local_repo_path.exists():
        msg = f"Local repository path does not exist: {local_repo_path}"
        logger.error(msg, file=str(local_repo_path))
        raise FileNotFoundError(msg)
    if not local_repo_path.is_dir():
        msg = f"Local repository path is not a directory: {local_repo_path}"
        logger.error(msg, file=str(local_repo_path))
        raise NotADirectoryError(msg)
    try:
        Repo(local_repo_path)
        logger.info("Validated Git repository.")
    except InvalidGitRepositoryError:
        logger.info("Path is not a Git repository; proceeding as source directory.")
    return local_repo_path.resolve()


def resolve_repository_path(
    use_git_url: bool, repo_url: str, local_repo_path: str | Path, workspace_dir: Path,
    if_clone_exists: str, logger: NotebookLogger, clone_depth: int | None = None,
) -> Path:
    if use_git_url:
        return clone_or_reuse_repository(repo_url, workspace_dir, if_clone_exists, logger, clone_depth)
    return validate_local_repo_path(Path(local_repo_path), logger)


def discover_javascript_files(repo_path: Path) -> list[Path]:
    extensions = {".js", ".mjs", ".cjs"}
    files: list[Path] = []
    for path in repo_path.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in extensions:
            continue
        if any(part in EXCLUDED_DIR_NAMES for part in path.parts):
            continue
        files.append(path.resolve())
    return sorted(files)


def compute_repository_stats(repo_path: Path, js_files: list[Path]) -> dict[str, Any]:
    total_size = sum(path.stat().st_size for path in js_files)
    directories = {path.parent for path in js_files}
    return {
        "repository_name": repo_path.name,
        "repository_size_bytes": total_size,
        "directory_count": len(directories),
        "javascript_file_count": len(js_files),
    }


def save_javascript_inventory(js_files: list[Path], output_csv: Path) -> None:
    pd.DataFrame(
        [{"file_path": str(p), "file_name": p.name, "directory": str(p.parent)} for p in js_files]
    ).to_csv(output_csv, index=False)


def preview_raw_output(raw_text: str, preview_lines: int, output_path: Path) -> None:
    lines = raw_text.splitlines()
    print(f"\n{'=' * 80}")
    print(f"RAW ESLINT OUTPUT PREVIEW (first {preview_lines} lines)")
    print(f"{'=' * 80}\n")
    if not lines:
        print("(empty raw output)")
        return
    print("\n".join(lines[:preview_lines]))
    remaining = len(lines) - preview_lines
    if remaining > 0:
        print(f"\n... ({remaining} more lines saved to {output_path})")

from run_comment_to_code_ratio_benchmark_impl import (
    build_comment_code_metrics,
    build_eslint_command,
    build_maintainability_summary,
    combine_raw,
    compute_comment_ratio,
    ensure_eslint_config,
    parse_eslint_json,
    records_to_findings,
    resolve_eslint_executable,
    run_command,
)



## Section 4 — Repository Setup


In [ ]:
OUTPUT_PATH = Path(OUTPUT_DIR).resolve()
WORKSPACE_PATH = Path(WORKSPACE_DIR).resolve()
ERROR_LOG_PATH = OUTPUT_PATH / 'error_log.txt'

ensure_output_dir(OUTPUT_PATH)
logger = NotebookLogger(ERROR_LOG_PATH)

try:
    REPO_PATH = resolve_repository_path(
        use_git_url=USE_GIT_URL, repo_url=REPO_URL, local_repo_path=LOCAL_REPO_PATH,
        workspace_dir=WORKSPACE_PATH, if_clone_exists=IF_CLONE_EXISTS, logger=logger, clone_depth=CLONE_DEPTH,
    )
except Exception as exc:
    logger.error(f'Repository setup failed: {exc}')
    raise

JS_FILES = discover_javascript_files(REPO_PATH)
if not JS_FILES:
    logger.error('No JavaScript source files found in repository.', file=str(REPO_PATH))
    raise FileNotFoundError('No JavaScript source files found.')

REPO_STATS = compute_repository_stats(REPO_PATH, JS_FILES)
ESLINT_EXE = resolve_eslint_executable(RUNTIMES_ROOT)
logger.info(f'Repository ready at: {REPO_PATH}')
logger.info(f'Using ESLint: {ESLINT_EXE}')
print(f"Repository: {REPO_STATS['repository_name']}")
print(f"Size (JavaScript files): {REPO_STATS['repository_size_bytes']:,} bytes")
print(f"Directories: {REPO_STATS['directory_count']:,}")
print(f"JavaScript files: {REPO_STATS['javascript_file_count']:,}")


## Section 5 — Discover JavaScript Files


In [ ]:
INVENTORY_CSV = OUTPUT_PATH / 'javascript_files_inventory.csv'
save_javascript_inventory(JS_FILES, INVENTORY_CSV)

print(f'Total JavaScript Files Found: {len(JS_FILES)}')
print(f'Saved inventory to: {INVENTORY_CSV}')


## Section 6 — Generate ESLint Configuration


In [ ]:
ESLINTRC_PATH = ensure_eslint_config(REPO_PATH)

print(f'Generated ESLint config: {ESLINTRC_PATH}')
print(ESLINTRC_PATH.read_text(encoding='utf-8'))


## Section 7 — Execute ESLint

Run ESLint in text and JSON formats. Preserve stdout/stderr exactly as emitted.


In [ ]:
text_cmd = build_eslint_command(ESLINT_EXE, REPO_PATH)
json_cmd = build_eslint_command(ESLINT_EXE, REPO_PATH, 'json')

raw_out, raw_err, raw_code = run_command(text_cmd)
json_out, json_err, json_code = run_command(json_cmd)

ESLINT_CONSOLE_CHUNKS = [
    '===== eslint (text) =====\n' + combine_raw(raw_out, raw_err),
    '===== eslint (json) =====\n' + combine_raw(json_out, json_err),
]

if raw_code not in {0, 1} and not json_out.strip():
    logger.error(f'ESLint text run exited with code {raw_code}', file='eslint_text')
if json_code not in {0, 1} and not json_out.strip():
    logger.error(f'ESLint JSON run exited with code {json_code}', file='eslint_json')

logger.info('ESLint execution complete.')


## Section 8 — Raw Output Extraction


In [ ]:
CONSOLE_PATH = OUTPUT_PATH / 'eslint_raw_console_output.txt'
JSON_PATH = OUTPUT_PATH / 'eslint_output.json'

CONSOLE_PATH.write_text('\n'.join(ESLINT_CONSOLE_CHUNKS), encoding='utf-8')
JSON_PATH.write_text(json_out, encoding='utf-8')

FINDINGS_DF = records_to_findings(parse_eslint_json(json_out))
FINDINGS_DF.to_csv(OUTPUT_PATH / 'eslint_findings.csv', index=False)

logger.info(f'Saved ESLint outputs and {len(FINDINGS_DF)} findings.')
preview_raw_output(CONSOLE_PATH.read_text(encoding='utf-8'), RAW_OUTPUT_PREVIEW_LINES, CONSOLE_PATH)


## Section 9 — Extract Comment and Code Metrics

Parse JavaScript source files for comment and executable code line counts.


In [ ]:
COMMENT_METRICS_DF = build_comment_code_metrics(JS_FILES)
COMMENT_METRICS_CSV = OUTPUT_PATH / 'comment_code_metrics.csv'
COMMENT_METRICS_DF.to_csv(COMMENT_METRICS_CSV, index=False)

logger.info(f'Extracted comment/code metrics for {len(COMMENT_METRICS_DF)} files.')
display(COMMENT_METRICS_DF)


## Section 10 — Comment-to-Code Ratio (Derived)

**Derived metric** (not emitted directly by ESLint):

```text
Total_Comment_Lines = comment_lines + block_comment_lines + single_comment_lines
Comment_to_Code_Ratio = Total_Comment_Lines / code_lines
```


In [ ]:
COMMENT_RATIO = compute_comment_ratio(COMMENT_METRICS_DF)
RATIO_CSV = OUTPUT_PATH / 'comment_to_code_ratio_summary.csv'
pd.DataFrame([
    {'metric_name': 'Comment_to_Code_Ratio', 'metric_value': COMMENT_RATIO['comment_to_code_ratio']},
]).to_csv(RATIO_CSV, index=False)

logger.info(f"Comment-to-Code Ratio={COMMENT_RATIO['comment_to_code_ratio']} (Derived)")
display(pd.read_csv(RATIO_CSV))


## Section 11 — Comment Percentage (Derived)


In [ ]:
PERCENTAGE_CSV = OUTPUT_PATH / 'comment_percentage_summary.csv'
pd.DataFrame([
    {'metric_name': 'Comment_to_Code_Percentage', 'metric_value': COMMENT_RATIO['comment_to_code_percentage']},
]).to_csv(PERCENTAGE_CSV, index=False)

logger.info(f"Comment Percentage={COMMENT_RATIO['comment_to_code_percentage']}%")
display(pd.read_csv(PERCENTAGE_CSV))


## Section 12 — Maintainability Summary


In [ ]:
MAINTAINABILITY_DF = build_maintainability_summary(FINDINGS_DF)
MAINTAINABILITY_CSV = OUTPUT_PATH / 'maintainability_summary.csv'
MAINTAINABILITY_DF.to_csv(MAINTAINABILITY_CSV, index=False)

display(MAINTAINABILITY_DF)


## Section 13 — Summary Dashboard


In [ ]:
doc_violations = int(MAINTAINABILITY_DF.loc[MAINTAINABILITY_DF['metric_name'] == 'Total_Documentation_Violations', 'metric_value'].iloc[0])
maint_violations = int(MAINTAINABILITY_DF.loc[MAINTAINABILITY_DF['metric_name'] == 'Total_Maintainability_Violations', 'metric_value'].iloc[0])

summary_df = pd.DataFrame([
    {'Metric': 'Total JavaScript Files', 'Value': len(JS_FILES)},
    {'Metric': 'Total Comment Lines', 'Value': int(COMMENT_RATIO['total_comment_lines'])},
    {'Metric': 'Total Code Lines', 'Value': int(COMMENT_RATIO['total_code_lines'])},
    {'Metric': 'Comment-to-Code Ratio (Derived)', 'Value': COMMENT_RATIO['comment_to_code_ratio']},
    {'Metric': 'Comment Percentage (Derived)', 'Value': COMMENT_RATIO['comment_to_code_percentage']},
    {'Metric': 'Documentation Violations', 'Value': doc_violations},
    {'Metric': 'Maintainability Violations', 'Value': maint_violations},
])
display(summary_df)

deliverables = [
    CONSOLE_PATH, JSON_PATH, OUTPUT_PATH / 'eslint_findings.csv', COMMENT_METRICS_CSV,
    RATIO_CSV, PERCENTAGE_CSV, MAINTAINABILITY_CSV, INVENTORY_CSV, ERROR_LOG_PATH,
    REPO_PATH / '.eslintrc.json',
]
print('\nDeliverables:')
for path in deliverables:
    print(f"  [{'OK' if path.exists() else 'MISSING'}] {path}")


## Section 14 — Error Handling


In [ ]:
if ERROR_LOG_PATH.exists() and ERROR_LOG_PATH.stat().st_size > 0:
    print(ERROR_LOG_PATH.read_text(encoding='utf-8'))
else:
    print('No errors logged.')


## Section 15 — Deliverables

```text
outputs/
├── eslint_raw_console_output.txt
├── eslint_output.json
├── eslint_findings.csv
├── comment_code_metrics.csv
├── comment_to_code_ratio_summary.csv
├── comment_percentage_summary.csv
├── maintainability_summary.csv
├── javascript_files_inventory.csv
└── error_log.txt
```
